# Rendimiento en Spark: Escalado y Gestión de Recursos

En este notebook exploramos cómo cambia el rendimiento de Spark cuando ajustamos los recursos de nuestro clúster. Utilizaremos una simulación de Monte Carlo para estimar π y mediremos cómo varía el tiempo de ejecución al escalar horizontal y verticalmente, y al limitar la memoria de los workers.

### Escenarios

| # | Escenario | Qué cambia | Concepto clave |
|---|-----------|-----------|----------------|
| 1 | **Referencia** | 1 worker, 1 core, 1 GB RAM | Punto de referencia |
| 2 | **Escalado horizontal** | 3 workers, 1 core cada uno, 1 GB RAM cada uno | Añadir más máquinas |
| 3 | **Escalado vertical** | 1 worker, 3 cores, 3 GB RAM | Potenciar una máquina |
| 4 | **Volcado a disco** | 1 worker, 3 cores, 512 MB RAM | La memoria importa |
| 5 | **Impacto del particionado** | 1 worker, 3 cores, 3 GB RAM, diferentes nº de particiones | Ajuste del paralelismo |

### Cómo funciona

Antes de cada escenario, deberás **reconfigurar el clúster Spark manualmente desde un terminal**, cambiando las variables de entorno de los workers y reiniciando los contenedores Docker. Cada escenario obtiene una SparkSession nueva para garantizar una medición justa.

**Requisitos previos**: El cluster de Hadoop debe estar desplegado (`docker compose up` en `hadoop/`).

---

## Funciones auxiliares

In [ ]:
import time
import random
from pyspark.sql import SparkSession


# ── Carga de trabajo: Monte Carlo π ──────────────────────────────

def estimate_pi(spark: SparkSession, num_samples: int, num_partitions: int = None) -> dict:
    """
    Estima π usando el método de Monte Carlo distribuido en Spark.
    Devuelve un dict con la estimación, el error, el tiempo y la configuración.
    """
    sc = spark.sparkContext

    if num_partitions is None:
        num_partitions = sc.defaultParallelism

    start = time.time()

    # Generar puntos aleatorios y contar cuántos caen dentro del círculo unitario
    count = sc.parallelize(range(num_samples), num_partitions) \
        .map(lambda _: (random.random(), random.random())) \
        .filter(lambda p: p[0]**2 + p[1]**2 <= 1) \
        .count()

    pi_estimate = 4.0 * count / num_samples
    elapsed = time.time() - start
    error = abs(pi_estimate - 3.141592653589793)

    result = {
        "app_name": spark.sparkContext.appName,
        "pi_estimate": round(pi_estimate, 6),
        "error": round(error, 6),
        "time_seconds": round(elapsed, 2),
        "num_samples": num_samples,
        "num_partitions": num_partitions,
        "parallelism": sc.defaultParallelism,
    }

    print(f"   π ≈ {result['pi_estimate']}  |  error = {result['error']}  |  ⏱ {result['time_seconds']}s  |  particiones = {result['num_partitions']}")
    return result


# Almacenar resultados para comparación
results = []
SAMPLES = 50_000_000  # 50 millones de puntos

print("Funciones auxiliares cargadas. ¡Listo para ejecutar los escenarios!")

---

## Escenario 1: Referencia mínima

Comenzamos con el clúster mínimo: **1 worker, 1 core, 1 GB de RAM**.

```
┌──────────────┐
│ spark-master │
└──────┬───────┘
       │
  ┌────┴─────┐
  │ worker-1 │  1 core, 1 GB
  └──────────┘
```

Este es nuestro punto de referencia y la configuración más lenta, ya que los 50 millones de muestras deben procesarse secuencialmente en un solo core.

### Antes de ejecutar configura el clúster desde un terminal

```bash
cd spark
docker compose down
WORKER_CORES=1 WORKER_MEMORY=1g docker compose up -d spark-worker-1
```

Espera unos segundos a que el worker se registre y verifica en http://localhost:8080.

In [ ]:
spark1 = SparkSession.builder \
    .appName("Escenario 1 - Referencia mínima") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter") \
    .getOrCreate()

print(f"📊 Ejecutando carga de trabajo: {SAMPLES:,} muestras en 1 worker × 1 core")
r1 = estimate_pi(spark1, SAMPLES, num_partitions=6)
results.append(r1)

spark1.stop()

---

## Escenario 2: Escalado horizontal

Ahora pasamos de 1 worker a 3 workers, cada uno con 1 core y 1 GB de RAM.

```
┌──────────────┐
│ spark-master │
└──────┬───────┘
       │
  ┌────┴─────┬──────────┬──────────┐
  │ worker-1 │ worker-2 │ worker-3 │
  │ 1c, 1 GB │ 1c, 1 GB │ 1c, 1 GB │
  └──────────┴──────────┴──────────┘
```

**Expectativa**: ~3× de aceleración respecto a la referencia, ya que los 50M de muestras pueden repartirse entre 3 workers procesando en paralelo. En la práctica la aceleración de una aplicación será menor a 3× debido a la sobrecarga de coordinación (planificación, shuffles, serialización de tareas).

### Antes de ejecutar:

```bash
docker compose down
WORKER_CORES=1 WORKER_MEMORY=1g docker compose up -d spark-worker-1 spark-worker-2 spark-worker-3
```

Espera unos segundos y verifica que los 3 workers estén registrados en http://localhost:8080.

In [ ]:
# ── Escenario 2: Escalado horizontal (3 workers, 1 core cada uno, 1 GB cada uno) ──

spark2 = SparkSession.builder \
    .appName("Escenario 2 - Escalado horizontal") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter") \
    .getOrCreate()

print(f"📊 Ejecutando carga de trabajo: {SAMPLES:,} muestras en 3 workers × 1 core")
r2 = estimate_pi(spark2, SAMPLES, num_partitions=6)
results.append(r2)

# Calcular aceleración
speedup = r1["time_seconds"] / r2["time_seconds"]
print(f"\n   ⚡ Aceleración vs referencia: {speedup:.2f}×")

spark2.stop()

---

## Escenario 3: Escalado vertical

En vez de añadir más máquinas, potenciamos una sola máquina con 3 cores y 3 GB de RAM.

```
┌──────────────┐
│ spark-master │
└──────┬───────┘
       │
  ┌────┴─────┐
  │ worker-1 │  3 cores, 3 GB
  └──────────┘
```

**Expectativa**: Aceleración similar al escalado horizontal (~3×), ya que seguimos teniendo 3 cores en total. Sin embargo, el escalado vertical evita la sobrecarga de red entre nodos, pues toda la comunicación es por memoria compartida. Por otro lado, tiene un techo: Los recursos máximos de la máquina.

### Horizontal y Vertical: ¿Cuándo elegir cada uno?

| | Horizontal | Vertical |
|---|---|---|
| **Ventajas** | Escalabilidad casi infinita, tolerancia a fallos | Sin sobrecarga de red, arquitectura más simple |
| **Desventajas** | Shuffles por red, coste de coordinación | Límites de hardware, punto único de fallo |
| **Ideal para** | Datasets muy grandes, producción | Datasets moderados, desarrollo |

### Antes de ejecutar:

```bash
docker compose down
WORKER_CORES=3 WORKER_MEMORY=3g docker compose up -d spark-worker-1
```

Espera unos segundos y verifica que hay 1 worker con 3 cores en http://localhost:8080.

In [ ]:
# ── Escenario 3: Escalado vertical (1 worker, 3 cores, 3 GB) ──

spark3 = SparkSession.builder \
    .appName("Escenario 3 - Escalado vertical") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter") \
    .getOrCreate()

print(f"📊 Ejecutando carga de trabajo: {SAMPLES:,} muestras en 1 worker × 3 cores")
r3 = estimate_pi(spark3, SAMPLES, num_partitions=6)
results.append(r3)

speedup = r1["time_seconds"] / r3["time_seconds"]
print(f"\n   ⚡ Aceleración vs referencia: {speedup:.2f}×")
print(f"   Escalado {'Vertical más rápido' if r3['time_seconds'] < r2['time_seconds'] else 'Horizontal más rápido'} por {abs(r3['time_seconds'] - r2['time_seconds']):.2f}s")

spark3.stop()

---

## Escenario 4: Volcado a disco cuando nos quedamos sin memoria (Disk Spill)

### ¿Qué es el volcado a disco?

Cuando Spark no tiene suficiente RAM para almacenar datos intermedios, vuelca (escribe) los datos a disco. La E/S a disco es órdenes de magnitud más lenta que la RAM:
- **RAM**: ~10 GB/s de throughput, ~100 ns de latencia
- **SSD**: ~0,5 GB/s de throughput, ~100 μs de latencia (1.000× más lento)
- **HDD**: ~0,1 GB/s de throughput, ~10 ms de latencia (100.000× más lento)

Esto significa que un trabajo de Spark sin memoria suficiente puede volverse catastróficamente lento.

Usaremos una carga de trabajo diferente aquí, una agregación GroupBy sobre un gran DataFrame que necesita hacer shuffle y ordenar datos en memoria. 

### Ejecutamos la referencia:

Primero ejecutaremos con suficiente memoria (3 GB, mantén la configuración del Escenario 3) para establecer una referencia:

In [ ]:
# ── Escenario 4a: GroupBy con suficiente memoria (3 GB) ──

spark4a = SparkSession.builder \
    .appName("Escenario 4a - GroupBy (3 GB)") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter") \
    .getOrCreate()

print("📊 Generando un gran DataFrame con 100M filas para agregación GroupBy...")

from pyspark.sql import functions as F

NUM_ROWS = 100_000_000

t0 = time.time()

# Crear un DataFrame con datos aleatorios: user_id (1-2000000), categoría (1-100), importe
df = spark4a.range(0, NUM_ROWS, numPartitions=10) \
    .withColumn("user_id", (F.rand() * 2000000).cast("int")) \
    .withColumn("category", (F.rand() * 100).cast("int")) \
    .withColumn("amount", F.rand() * 1000)

# Forzar un shuffle: agrupar por user_id, calcular agregaciones
agg_df = df.groupBy("user_id").agg(
    F.count("*").alias("num_transactions"),
    F.sum("amount").alias("total_amount"),
    F.avg("amount").alias("avg_amount"),
    F.max("amount").alias("max_amount"),
)

# Ejecutar la acción
row_count = agg_df.count()
t1 = time.time()

time_with_memory = round(t1 - t0, 2)
print(f"   ✅ Agregadas {NUM_ROWS:,} filas → {row_count:,} grupos en {time_with_memory}s (con 3 GB de RAM)")

spark4a.stop()

### Ejecutamos con memoria limitada:

Ahora reducimos la memoria del executor a solo 512 MB de RAM y volvemos a lanzar.

Podemos ver la interfaz de la sesión en ejecución en http://localhost:4040. Observad **Spill (Memory)** y **Spill (Disk)** en la pestaña **Stages**.

In [ ]:
# ── Escenario 4b: Mismo GroupBy con memoria limitada (512 MB) ──

spark4b = SparkSession.builder \
    .appName("Escenario 4b - GroupBy (512 MB)") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter") \
    .config("spark.executor.memory", "512m") \
    .getOrCreate()

print("📊 Misma carga de trabajo, pero ahora con solo 512 MB de RAM...")

from pyspark.sql import functions as F

t0 = time.time()

df = spark4b.range(0, NUM_ROWS, numPartitions=10) \
    .withColumn("user_id", (F.rand() * 2000000).cast("int")) \
    .withColumn("category", (F.rand() * 100).cast("int")) \
    .withColumn("amount", F.rand() * 1000)

agg_df = df.groupBy("user_id").agg(
    F.count("*").alias("num_transactions"),
    F.sum("amount").alias("total_amount"),
    F.avg("amount").alias("avg_amount"),
    F.max("amount").alias("max_amount"),
)

row_count = agg_df.count()
t1 = time.time()

time_without_memory = round(t1 - t0, 2)
print(f"   ⚠️  Agregadas {NUM_ROWS:,} filas → {row_count:,} grupos en {time_without_memory}s (con solo 512 MB de RAM)")
print(f"\n   🐌 Ralentización por falta de memoria: {time_without_memory / time_with_memory:.2f}×")

In [ ]:
# ¡Recordad cerrar la sesión!
spark4b.stop()

---

## Escenario 5: El impacto del particionado

Incluso con el mismo hardware, **cómo divides los datos** (particiones) afecta drásticamente al rendimiento.

- **Muy pocas particiones** → Los cores quedan ociosos, el trabajo está desbalanceado.
- **Demasiadas particiones** → Sobrecarga excesiva de planificación, tareas diminutas.
- **El número justo** → Todos los cores se mantienen ocupados con fragmentos significativos de trabajo

Ejecutaremos nuestra carga Monte Carlo π con 3 workers × 1 core pero variando el número de particiones.

**Regla general**: Usa 2-4 particiones por core disponible como punto de partida.

### Antes de ejecutar, configura el clúster con sus parámetros por defecto (archivo `.env`):

```bash
docker compose down
docker compose up -d
```

Espera unos segundos y verifica que hay 3 workers en http://localhost:8080.

In [ ]:
# ── Escenario 5: Impacto del particionado (3 workers, 1 core cada uno) ──

spark5 = SparkSession.builder \
    .appName("Escenario 5 - Particionado") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter") \
    .getOrCreate()

partition_counts = [1, 2, 3, 4, 6, 12, 50, 200]
partition_results = []

print(f"📊 Probando {len(partition_counts)} configuraciones de particiones en 3 workers × 1 core\n")

for np in partition_counts:
    print(f"   Particiones = {np:3}:", end="")
    r = estimate_pi(spark5, SAMPLES, num_partitions=np)
    r["label"] = f"{np} particiones"
    partition_results.append(r)

spark5.stop()

print(f"\nMejor nº de particiones: {min(partition_results, key=lambda x: x['time_seconds'])['num_partitions']}")
print(f"Peor nº de particiones: {max(partition_results, key=lambda x: x['time_seconds'])['num_partitions']}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

part_labels = [str(r["num_partitions"]) for r in partition_results]
part_times = [r["time_seconds"] for r in partition_results]

ax.plot(part_labels, part_times, "o-", color="#9b59b6", linewidth=2, markersize=8)
ax.fill_between(range(len(part_labels)), part_times, alpha=0.15, color="#9b59b6")
ax.set_xlabel("Número de particiones")
ax.set_ylabel("Tiempo de ejecución (segundos)")
ax.set_title("Monte Carlo π — Impacto del particionado\n(3 workers × 1 core)")

# Anotar el mejor resultado
best_idx = part_times.index(min(part_times))
ax.annotate(f"Mejor: {part_times[best_idx]}s",
            xy=(best_idx, part_times[best_idx]),
            xytext=(best_idx + 0.5, part_times[best_idx] + max(part_times)*0.1),
            arrowprops=dict(arrowstyle="->", color="#27ae60"),
            fontsize=11, color="#27ae60", fontweight="bold")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

---

## Conclusiones

### 1. Escalado horizontal (añadir nodos)
- Añadir workers distribuye la carga de trabajo entre más máquinas.
- La aceleración es **sub-lineal** debido a la sobrecarga de coordinación (Ley de Amdahl).
- Ideal cuando el dataset es muy grande, se necesita tolerancia a fallos, entornos en la nube.

### 2. Escalado vertical (nodos más potentes)
- Más cores en una sola máquina evita la sobrecarga de shuffles por red.
- Toda la comunicación ocurre mediante memoria compartida (más rápida que la red).
- Ideal cuando tienes datasets moderados, desarrollo en una sola máquina, menor latencia.
- Una única máquina tiene un límite de potencia determinado por el hardware disponible.

### 3. La memoria importa
- Spark es un motor de procesamiento **en memoria**, esa es su fortaleza.
- Cuando la RAM es insuficiente, Spark vuelca a disco.
- Asegura siempre que `spark.executor.memory` sea lo suficientemente grande para tu carga de trabajo.

### 4. El particionado es rendimiento gratis
- Las particiones controlan el paralelismo: Cada partición es una tarea.
- Muy pocas particiones → Cores desperdiciados (infrautilización).
- Demasiadas particiones → Sobrecarga excesiva (sobreparalelización).
- Como regla, 2-4 particiones por core disponible.
---